# Pad Mask 与 Causal Mask 的原理与作用

## 一、问题背景：softmax 让每个位置"看见一切"

缩放点积注意力：

$$\text{Attn}(Q,K,V)=\text{softmax}\Big(\frac{QK^\top}{\sqrt{d_k}}\Big)V$$

点积 $QK^\top$ 得到 $L_q\times L_k$ 的分数矩阵，softmax 会给**每个 key 位置**分配非零权重。但有两类位置**不该被注意**：

| 问题 | 不该看的位置 | 对应的 mask |
| --- | --- | --- |
| batch 内句长不一，短句用 `<pad>` 补齐 | 填充 token | **Pad mask** |
| 自回归生成：第 $t$ 步只能依赖前 $t$ 个词 | 未来 token | **Causal mask** |

## 二、统一机制：softmax 前加"极负"偏移

$$\text{Attn}(Q,K,V)=\text{softmax}\Big(\frac{QK^\top}{\sqrt{d_k}}+M\Big)V,\qquad
M_{ij}=\begin{cases}0,&\text{位置 }j\text{ 允许被 }i\text{ 注意}\\-\infty,&\text{禁止}\end{cases}$$

- 原理：$e^{-\infty}=0$，被屏蔽位置在 softmax 后权重**严格为 0**，不参与 $V$ 的加权求和；其余位置权重相对比例不变。
- 下文用 ✓ 表示 $M_{ij}=0$（允许），✗ 表示 $M_{ij}=-\infty$（屏蔽）。

## 三、Pad Mask（填充遮罩）

### 1）Padding 的作用与填充规则

- **核心目的**：满足 Transformer 并行计算对矩阵形状的要求——把不同长度的输入文本统一填充到相同长度。
- **默认规则**：一般填充到同一批次中最长文本的长度；填充位置集中在文本末尾（右填充约定）。
- **为什么需要填充**：不同文本天然长度不一致，不填充就无法形成统一形状的矩阵，无法进行批量矩阵乘法。

### 2）为什么必须加 Mask：作用原理

- **Padding 本身无实际语义**：填充内容不含有效语义，但会参与注意力分数计算，因此必须对它做掩码处理。
- **Padding 不影响线性层计算**：线性层（$W_Q/W_K/W_V$ 变换）逐位置独立作用，每个 token 的向量不与其他 token 交叉，所以 Padding 不会改变有效 token 的线性变换结果。
- **Padding 的影响只发生在注意力计算阶段**：只有在 $QK^\top$ 计算相关分数时，有效 token 才会与 Padding token 产生关联——因此 Padding Mask 必须作用在注意力分数上。
- 屏蔽之后：pad 位置的权重严格为 0（$e^{-\infty}=0$），其 embedding 不再污染任何输出；同一句话无论 padding 多少，输出保持一致（batch invariance）。

### 3）具体实现逻辑

- **Padding 位置的特征**：在注意力得分矩阵中，Padding 只会出现在最后几列（作为 key）和最后几行（作为 query），不会出现在开头或中间（右填充约定）。
- **Mask 位置的获取方式**：原始输入 shape 为 $[B,S]$（$B$ 为批次句子数，$S$ 为统一后的句长），Padding 位置与输入中等于 `<pad_id>` 的位置完全一致，可直接从原始输入取得：`pad_mask = (input_ids == pad_id)`（True = 需要屏蔽）。
- **列的掩码方式**：把需要掩码的列（key 方向）所有注意力得分置为 $-\infty$，整理成 $[B,1,1,S]$ 广播到所有 query 行；Softmax 后这些位置的概率恰好为 0，从而消除 Padding 对有效位置输出的影响：

```text
keys:          我    爱    你   <pad>  <pad>
任一 query 行:  ✓     ✓     ✓     ✗      ✗
```

- **注意**：mask 屏蔽的是**作为 key 的位置**。pad 作为 query 时自身输出无意义，但它作为 key 始终被屏蔽，因此不会影响其他位置的输出；行方向（query）一般不掩——原因见下一小节。

### 4）最后几行 Padding 不处理的原因

- **整行全掩码会破坏 Softmax 归一化**：若把该行得分全部置为 $-\infty$，分母 $\sum_j e^{s_{ij}}=0$，出现 $0/0$ 的未定义情形——实际表现是**静默得到 NaN**（并非抛出异常），并沿后续计算传播。（若用 `-1e9` 这类有限负数代替 $-\infty$，整行会 Softmax 成均匀分布——同样是错误结果，只是更隐蔽。）
- **Padding 的影响被"锁"在 Padding 行内**：注意力输出按行独立——softmax 行内归一化后，输出的第 $i$ 行 = 该行权重对 $V$ 的加权和；后续线性层又逐位置作用。所以 Padding 行产生的输出永远不会泄漏到有效 token 的行，不需要提前处理。
- **最终的处理方式**：在输出头拿到结果后再屏蔽——推理时把对应位置的概率置 0；训练时用 loss mask 忽略这些位置的损失即可。

> 主流实现也是这个约定：PyTorch 的 `key_padding_mask` 与 HuggingFace 的 `attention_mask` 扩展都只在 key 方向屏蔽。

## 四、Causal Mask（因果遮罩 / 前瞻遮罩）

**原理**：下三角允许、上三角屏蔽的 $L\times L$ 矩阵，形状 $[1,1,L,L]$。位置 $i$ 只能看到 $\le i$ 的位置：

```text
         k0   k1   k2   k3
q0:      ✓    ✗    ✗    ✗
q1:      ✓    ✓    ✗    ✗
q2:      ✓    ✓    ✓    ✗
q3:      ✓    ✓    ✓    ✓
```

$$M_{\text{causal}}=\begin{bmatrix}
0&-\infty&-\infty&-\infty\\
0&0&-\infty&-\infty\\
0&0&0&-\infty\\
0&0&0&0
\end{bmatrix}$$

**作用**：
1. **自回归性**：第 $t$ 个位置的输出只依赖 $\le t$ 的输入；
2. **训练/推理一致**：训练时整句并行前向，每个位置的"可见范围"等价于推理时逐词生成看到的历史，所以一次前向即可算出全句 loss；
3. **防标签泄漏**：预测下一个词时不能提前看到答案。

## 五、两者叠加（Transformer Decoder 自注意力）

同一层里两种 mask 同时生效：causal 去掉右上三角，pad 去掉 `<pad>` 列。例：真实长度 3、pad 到长度 5：

```text
         k0   k1   k2  pad  pad
q0:      ✓    ✗    ✗    ✗    ✗
q1:      ✓    ✓    ✗    ✗    ✗
q2:      ✓    ✓    ✓    ✗    ✗    ← 真实 token 行：causal ∩ pad
q3:      ✓    ✓    ✓    ✗    ✗    ← pad 自身的行：输出无意义，训练时由 loss mask 忽略
q4:      ✓    ✓    ✓    ✗    ✗
```

- 实现上把两个布尔 mask 按位 OR（数值矩阵则相加），一次 `masked_fill` 完成；
- 补充：BERT 类双向编码器只用 pad mask；GPT 类只用 causal（+pad）；原版 Transformer decoder 自注意力两者叠加。Decoder 的**交叉注意力**对 encoder 输出只用 pad mask（query 在解码侧，不存在"未来"问题）。

## 六、PyTorch 实现要点

```python
# causal：严格上三角（不含对角线）为 True
causal = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)   # [L, L]

# pad：token id 等于 pad_id 的位置为 True
pad = (ids == pad_id)                                                 # [B, L_k]

# 组合并广播到 [B, 1, L_q, L_k]
mask = causal | pad[:, None, None, :]

scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)     # [B, H, L_q, L_k]
scores = scores.masked_fill(mask, float("-inf"))
attn = torch.softmax(scores, dim=-1)                  # 屏蔽位置权重 → 0
```

- 用 $-\infty$ 数学上精确；若用 `-1e9` 近似，注意 fp16 数值范围（最大约 65504），可改用 `-1e4`；
- 不要对 query（行）方向做整行屏蔽——后果与处理方式见「三、4」。

## 七、总结

- **Pad mask**：屏蔽"凑长度"的无效位置 → 解决*数据对齐引入的噪声*；
- **Causal mask**：屏蔽"未来"位置 → 解决*自回归生成的信息泄漏*；
- 两者本质相同——softmax 之前把不该看的位置打成 $-\infty$——只是作用位置不同，可以叠加使用。

Transformer 与位置编码


## 一、Transformer 推理流程

### 1. Encoder–Decoder 翻译模型

翻译任务中的推理过程如下：

    源句 src
      → Encoder 只计算一次
      → 得到 encoder memory，形状为 [B,S,D]
      → Decoder 从 <SOS> 开始自回归生成
      → 每次把新生成的 token 作为下一步输入
      → 生成 <EOS> 后停止

需要区分两类可复用信息：

- **Decoder Self-Attention 的 KV Cache**：过去 token 的 K、V 被逐步缓存，生成新 token 时只计算新增位置的 K、V。
- **Cross-Attention 的 Encoder K、V**：Encoder memory 在整个解码过程中保持不变，各层 Cross-Attention 可以预先计算并重复使用对应的 K、V。

对于 Decoder-only 大语言模型，没有 Encoder，KV Cache 来自 Decoder 各层的 Self-Attention。

### 2. 推理优化目标

自回归生成必须按 token 串行执行，因此推理优化主要关注：

- 减少历史 K、V 的重复计算：**KV Cache**；
- 降低 Attention 的显存读写开销：**FlashAttention**；
- 更高效地管理不同请求的 KV Cache 内存：**PagedAttention**；
- 进一步结合量化、算子融合、连续批处理和推测解码等工程方法。

FlashAttention 改善的是 Attention 的 IO 与显存效率，不会改变标准全注意力的数学结果。

## 二、Transformer 核心模块复习

### 1. 为什么通常使用 LayerNorm

Transformer 张量通常写成：

$$X\in\mathbb{R}^{B\times S\times D}$$

标准 LayerNorm 对每个 token 的最后一个特征维度 $D$ 计算均值和方差：

$$\mu_{b,s}=\frac{1}{D}\sum_{d=1}^{D}X_{b,s,d}$$

$$\sigma^2_{b,s}=\frac{1}{D}\sum_{d=1}^{D}(X_{b,s,d}-\mu_{b,s})^2$$

也就是说，LayerNorm 归一化的是 $X[b,s,:]$，**不是 sequence 维度 $S$**。

LayerNorm 更适合 Transformer 的原因包括：

- 不依赖同一个 batch 内的其他样本；
- 小 batch 下仍然稳定；
- 不需要 BatchNorm 的运行均值和运行方差；
- 适合变长序列、动态 padding 和自回归生成；
- 训练与推理阶段的归一化方式一致。

形状始终保持不变：

$$[B,S,D]\rightarrow[B,S,D]$$

### 2. Feed Forward Network 的作用

Position-wise FFN 对每个 token 独立使用同一组两层 MLP：

$$\operatorname{FFN}(x)=W_2\,\sigma(W_1x+b_1)+b_2$$

形状变化：

$$[B,S,D]\rightarrow[B,S,D_{ff}]\rightarrow[B,S,D]$$

其主要作用是：

1. 在特征维度上进行通道混合；
2. 通过激活函数引入更强的非线性；
3. 先升维再压缩，提高模型表示容量；
4. 对 Attention 汇总得到的信息进行逐 token 加工。

注意：Attention 内部的 Softmax 本身也是非线性运算，因此不能简单地说“Attention 完全没有非线性”。更准确的理解是：

> Attention 负责 token 之间的信息交换，FFN 负责每个 token 内部特征的非线性变换。

### 3. 输出头

Decoder 最终隐藏状态形状为：

$$H\in\mathbb{R}^{B\times T\times D}$$

通过线性输出头映射到词表维度：

$$\text{logits}=HW_{vocab}+b$$

$$[B,T,D]\rightarrow[B,T,V]$$

训练时通常直接把 logits 传入 CrossEntropyLoss，不需要手动执行 Softmax，因为交叉熵内部已经完成等价的 LogSoftmax 与负对数似然计算。推理时只有在需要概率采样时才显式计算概率；如果只取 argmax，直接比较 logits 即可。

## 三、为什么需要位置编码

Self-Attention 的输入可以整句并行计算。若不加入任何位置信息，模型无法仅凭 token 集合区分：

- “我爱你”；
- “你爱我”。

两句话包含相同 token，但顺序不同、语义不同。因此必须把位置信息引入 Transformer。

常见方法包括：

- **学习式绝对位置编码**：使用有限大小的位置 Embedding 表；
- **固定正余弦位置编码**：通过公式计算每个位置的多频率表示；
- **相对位置偏置**：直接根据 token 间距离修正注意力分数；
- **RoPE**：使用绝对位置计算旋转相位，使 Q、K 点积具有相对位置性质。

## 四、固定正余弦位置编码

### 1. 公式

$$PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/D}}\right)$$

$$PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/D}}\right)$$

其中：

- $pos$：token 的位置；
- $D=d_{model}$：模型特征维度；
- $0\le i<D/2$：第 $i$ 个正余弦维度对。

令：

$$\omega_i=10000^{-2i/D}$$

则每个维度对可以写为：

$$[\sin(pos\,\omega_i),\ \cos(pos\,\omega_i)]$$

不同维度对使用不同频率，因此位置同时具有高频的局部变化和低频的长距离变化。

### 2. 加入 Transformer 的方式

位置编码表的形状通常为：

$$[L,D]\rightarrow[1,L,D]$$

利用 batch 广播：

$$X=Embedding+PE$$

$$[B,L,D]+[1,L,D]\rightarrow[B,L,D]$$

### 3. 长度与唯一性的准确表述

- 学习式绝对位置 Embedding 通常具有固定的最大长度；超出表大小后没有对应参数。
- 固定正余弦编码可以通过公式计算训练长度以外的位置，因此没有查找表的硬上限。
- “可以计算超长位置”不等于“模型在超长序列上仍然表现良好”；训练长度之外的效果仍可能下降。
- 单独一对正余弦具有周期性。完整位置向量使用多组不同频率，使位置在实际上下文范围内具有可区分的多尺度表示，但不能仅凭“频率不是整数”就证明它在无限范围内绝不重复。

## 五、RoPE 的核心思想

RoPE 不把位置向量直接加到输入 Embedding 上，而是在每层 Attention 中：

    X
      → 线性投影得到 Q、K、V
      → 拆分多头
      → 对 Q、K 应用位置相关旋转
      → 计算 QKᵀ / √dₖ
      → Mask → Softmax → 乘 V

通常只旋转 Q、K，不旋转 V，因为位置信息需要直接影响的是注意力权重 $QK^\top$；V 负责承载随后被加权汇总的内容信息。

更严谨地说，RoPE 使用绝对位置 $p$ 计算旋转相位 $p\theta_i$，但旋转后的 Q、K 点积只依赖位置差，因此具有相对位置性质。

## 六、二维旋转矩阵

二维向量：

$$x=\begin{bmatrix}x_1\\x_2\end{bmatrix}$$

旋转角度 $\phi$ 后：

$$R(\phi)=
\begin{bmatrix}
\cos\phi&-\sin\phi\\
\sin\phi&\cos\phi
\end{bmatrix}$$

$$R(\phi)x=
\begin{bmatrix}
x_1\cos\phi-x_2\sin\phi\\
x_1\sin\phi+x_2\cos\phi
\end{bmatrix}$$

旋转矩阵具有两个关键性质：

### 1. 正交性

$$R(\phi)^TR(\phi)=I$$

因此：

$$R(\phi)^T=R(\phi)^{-1}=R(-\phi)$$

旋转只改变方向，不改变向量长度。

### 2. 角度可加性

$$R(\alpha)R(\beta)=R(\alpha+\beta)$$

这是旋转矩阵的群性质，可由三角函数和角公式验证。

## 七、RoPE 的相对位置推导

对位置 $p$ 的 Query 和位置 $m$ 的 Key，分别执行旋转：

$$q'_p=R(p\theta_i)q_p$$

$$k'_m=R(m\theta_i)k_m$$

旋转后的点积为：

$$
\begin{aligned}
(q'_p)^Tk'_m
&=(R(p\theta_i)q_p)^T(R(m\theta_i)k_m)\\
&=q_p^TR(p\theta_i)^TR(m\theta_i)k_m\\
&=q_p^TR(-p\theta_i)R(m\theta_i)k_m\\
&=q_p^TR((m-p)\theta_i)k_m
\end{aligned}
$$

最终只剩相对位置：

$$\boxed{(q'_p)^Tk'_m=q_p^TR((m-p)\theta_i)k_m}$$

如果 Q、K 的位置同时平移 $c$：

$$((m+c)-(p+c))=m-p$$

所以相对距离不变，旋转后的点积也保持不变。

把二维结果进一步展开，令 $\Delta=(m-p)\theta_i$：

$$q^TR(\Delta)k=(q_1k_1+q_2k_2)\cos\Delta+(q_2k_1-q_1k_2)\sin\Delta$$

这说明 RoPE 不是简单地给分数加一个距离常数，而是让内容关系与相对位置发生乘法耦合。

## 八、高维 RoPE

对于每个注意力头中的 $d_k$ 维向量，将最后一个维度两两分组：

    (0,1)、(2,3)、(4,5)、…、(dₖ−2,dₖ−1)

第 $i$ 对维度使用频率：

$$\theta_i=10000^{-2i/d_k}$$

位置 $p$ 对应的旋转角为：

$$p\theta_i$$

例如 $d_k=32$ 时，每个头有 16 个二维维度对。旋转前后形状不变：

$$[B,H,L,d_k]\rightarrow[B,H,L,d_k]$$

随后计算：

$$Q_{rope}K_{rope}^T\rightarrow[B,H,L_q,L_k]$$

## 九、为什么在 Q、K 投影后旋转

如果先旋转输入，再经过任意投影矩阵 $W_Q、W_K$，通常有：

$$W_QR_p\ne R_pW_Q$$

也就是说，任意线性投影一般不与旋转矩阵交换，无法保证后续点积仍能清晰化简为 $R_{m-p}$。

在得到 Q、K 后再旋转，则可以直接利用：

$$R_p^TR_m=R_{m-p}$$

从而保留严格的相对位置结构。

## 十、RoPE 的工程实现

### 1. 构造频率与角度

    inv_freq = 1 / 10000 ** (arange(0, rotary_dim, 2) / rotary_dim)
    angles = position[:, None] * inv_freq[None, :]

形状为：

$$[L,1]\times[1,d_{rope}/2]\rightarrow[L,d_{rope}/2]$$

然后计算对应的 cos 和 sin，并扩展为能够与 Q、K 广播的形状。

### 2. 避免构造稀疏分块矩阵

定义二维置换：

$$rotate\_half([x_1,x_2])=[-x_2,x_1]$$

则二维旋转可以写成：

$$x'=x\cos\phi+rotate\_half(x)\sin\phi$$

对应代码思想：

    x_even = x[..., 0::2]
    x_odd  = x[..., 1::2]
    x_rot  = stack([-x_odd, x_even], dim=-1).flatten(-2)
    x_rope = x * cos + x_rot * sin

这样不需要创建包含大量零元素的分块对角矩阵。

### 3. cos/sin 缓存

常见流程是：

1. 首次根据初始最大长度建立 cos/sin 缓存；
2. 当前序列不超过缓存时直接切片；
3. 超过缓存长度时扩容，而不是每次 forward 都重建全部缓存。

register_buffer 的准确含义：

- buffer 不参与梯度更新，也不会进入优化器；
- buffer 会跟随模型自动迁移设备；
- 默认 persistent=True，因此会进入 state_dict；
- 若不希望缓存写入 checkpoint，应设置 persistent=False。

### 4. 部分维度旋转

实际模型可以只对 Q、K 的部分维度应用 RoPE，例如 head_dim=128、rotary_dim=64。这样能够降低旋转计算量，并保留一部分未旋转的内容维度。但这是位置建模能力与计算开销之间的设计权衡，不能笼统表述为“完全不影响效果”。

## 十一、RoPE 的能力边界

- RoPE 让注意力点积自然依赖相对位置，但仍然使用绝对位置生成旋转相位。
- RoPE 不会代替 Pad Mask 或 Causal Mask；Decoder 仍然需要 Causal Mask 防止查看未来 token。
- 标准 RoPE 不自动保证训练长度之外的性能。长上下文模型通常还需要 Position Interpolation、NTK-aware scaling、Dynamic NTK、YaRN 等缩放策略及相应训练适配。
- RoPE 改变的是 Q、K 的数值，不改变张量形状和 Attention 的基本计算接口。

## 十二、本节课结论

1. Transformer 推理的主要瓶颈来自自回归串行生成，KV Cache、FlashAttention 和 PagedAttention 是重要的工程优化方向。
2. LayerNorm 对每个 token 的特征维度 $D$ 做归一化，不是在 sequence 维度上计算。
3. FFN 负责逐 token 的非线性特征变换；输出头将 $D$ 维隐藏状态映射为 $V$ 维 logits。
4. 固定正余弦编码通过多组频率给输入加入绝对位置表示，但能够计算更长位置不等于模型一定具备超长外推能力。
5. RoPE 在 Attention 内旋转 Q、K，通过 $R_p^TR_m=R_{m-p}$ 使点积具有相对位置性质。
6. 工程实现通过 cos/sin 与逐元素运算完成旋转，无需显式构造大型旋转矩阵。
7. 课程末布置了相关作业，但当前录音总结未包含具体题目与提交要求，应以教师原始通知为准，不在此补写。

---

下面继续记录 Pad Mask 与 Causal Mask 专题。
